In [1]:
import sectio.providers.sqlite_provider as prov
from sectio.core import CrossSection

print(f"CrossSection class: {CrossSection}")
# This should print: <class 'sectio.core.CrossSection'>

CrossSection class: <class 'sectio.core.CrossSection'>


In [2]:
import pandas as pd
import sectio
from tqdm import tqdm

all_families = sectio.catalog.list_families()
results = [] # Ensure this is reset

for family in tqdm(all_families, desc="Auditing Families"):
    if family == "DATA_DICTIONARY": 
        continue
    try:
        df_family = sectio.catalog.get_family(family)
        table_name = f"sections_{family.lower()}"
        
        for sid in df_family['Section_ID'].tolist():
            try:
                cs = sectio.db.get_section(table_name, sid, subdivision='calc')
                
                # Extract and Append
                results.append({
                    "Family": family,
                    "Section_ID": sid,
                    "DB_Area": float(cs.metadata.get('A', 0.0)),
                    "Calc_Area": cs.area,
                    "DB_Iy": float(cs.metadata.get('Iy', 0.0)),
                    "Calc_Iy": cs.Iy,
                    "DB_Iz": float(cs.metadata.get('Iz', 0.0)),
                    "Calc_Iz": cs.Iz,
                    "DB_J": float(cs.metadata.get('It', 0.0)), # Note: DB uses 'It'
                    "Calc_J": cs.J
                })
            except Exception as e:
                # Log the specific error if a section fails
                print(f"Error in {family} | {sid}: {e}")
                continue
    except Exception as e:
        print(f"Failed to load family {family}: {e}")
        continue

audit_df = pd.DataFrame(results)
print(f"Audit complete! {len(audit_df)} rows captured.")

Auditing Families: 100%|██████████| 22/22 [00:01<00:00, 12.58it/s]

Audit complete! 1020 rows captured.


In [3]:
# 1. Calculate Absolute Differences
audit_df['Area_Diff'] = audit_df['Calc_Area'] - audit_df['DB_Area']
audit_df['Iy_Diff'] = audit_df['Calc_Iy'] - audit_df['DB_Iy']
audit_df['Iz_Diff'] = audit_df['Calc_Iz'] - audit_df['DB_Iz']

# 2. Calculate Percentage Errors (Relative to DB values)
# Using a small epsilon to avoid division by zero
epsilon = 1e-9
audit_df['Area_Err%'] = (abs(audit_df['Area_Diff']) / (audit_df['DB_Area'] + epsilon)) * 100
audit_df['Iy_Err%'] = (abs(audit_df['Iy_Diff']) / (audit_df['DB_Iy'] + epsilon)) * 100
audit_df['Iz_Err%'] = (abs(audit_df['Iz_Diff']) / (audit_df['DB_Iz'] + epsilon)) * 100

# 3. Round for readability
audit_df = audit_df.round(4)

print("✅ Error metrics calculated.")

✅ Error metrics calculated.


In [4]:
audit_df.columns

Index(['Family', 'Section_ID', 'DB_Area', 'Calc_Area', 'DB_Iy', 'Calc_Iy',
       'DB_Iz', 'Calc_Iz', 'DB_J', 'Calc_J', 'Area_Diff', 'Iy_Diff', 'Iz_Diff',
       'Area_Err%', 'Iy_Err%', 'Iz_Err%'],
      dtype='str')

In [5]:
# Create a dedicated inspection dataframe for Area
area_audit = audit_df[['Family', 'Section_ID', 'DB_Area', 'Calc_Area', 'Area_Diff', 'Area_Err%']].copy()

# Sort by the most egregious errors
area_audit = area_audit.sort_values('Area_Err%', ascending=False)

print("Top 10 Area Discrepancies:")
print(area_audit.head(10))

# Summary statistics for the 'Sanity Check'
print(f"\nMean Area Error: {area_audit['Area_Err%'].mean():.4f}%")
print(f"Max Area Error:  {area_audit['Area_Err%'].max():.4f}%")

Top 10 Area Discrepancies:
    Family Section_ID  DB_Area  Calc_Area  Area_Diff  Area_Err%
319     UE       UE50    616.0   605.8118   -10.1882     1.6539
323     UE      UE120   1330.0  1308.3541   -21.6459     1.6275
320     UE       UE65    751.0   738.8678   -12.1322     1.6155
321     UE       UE80    898.0   883.5364   -14.4636     1.6106
327     UE      UE200   2340.0  2307.0756   -32.9244     1.4070
331     UE      UE300   4050.0  3993.6680   -56.3320     1.3909
326     UE      UE180   2070.0  2041.3690   -28.6310     1.3831
325     UE      UE160   1810.0  1785.5065   -24.4935     1.3532
328     UE      UE220   2670.0  2635.3715   -34.6285     1.2969
330     UE      UE270   3520.0  3475.9616   -44.0384     1.2511

Mean Area Error: 0.0834%
Max Area Error:  1.6539%


In [6]:
# 1. Filter out rows with zero database values to ensure valid percentages
active_audit = audit_df[
    (audit_df['DB_Area'] > 0) & 
    (audit_df['DB_Iy'] > 0) & 
    (audit_df['DB_Iz'] > 0)
].copy()

# 2. Define standard columns for display
cols = ['Family', 'Section_ID', 'Area_Err%', 'Iy_Err%', 'Iz_Err%']

print("🚨 TOP 10 AREA DISCREPANCIES")
print("Check for: Incorrect wall thickness, missing radii, or unit mismatches.")
print("-" * 85)
print(active_audit.sort_values('Area_Err%', ascending=False)[cols].head(10).to_string(index=False))

print("\n\n🚨 TOP 10 Iy DISCREPANCIES (Strong Axis)")
print("Check for: Flange thickness discrepancies or depth (h) mismatches.")
print("-" * 85)
print(active_audit.sort_values('Iy_Err%', ascending=False)[cols].head(10).to_string(index=False))

print("\n\n🚨 TOP 10 Iz DISCREPANCIES (Weak Axis)")
print("Check for: Taper slope (9% vs 5%) or width (b) mismatches.")
print("-" * 85)
print(active_audit.sort_values('Iz_Err%', ascending=False)[cols].head(10).to_string(index=False))

🚨 TOP 10 AREA DISCREPANCIES
Check for: Incorrect wall thickness, missing radii, or unit mismatches.
-------------------------------------------------------------------------------------
Family Section_ID  Area_Err%  Iy_Err%  Iz_Err%
    UE       UE50     1.6539   1.6549   1.2071
    UE      UE120     1.6275   1.7639   0.9812
    UE       UE65     1.6155   1.5335   1.1445
    UE       UE80     1.6106   1.5701   1.2196
    UE      UE200     1.4070   1.5612   0.0516
    UE      UE300     1.3909   1.7444   0.3425
    UE      UE180     1.3831   2.0336   0.4962
    UE      UE160     1.3532   1.7143   0.6044
    UE      UE220     1.2969   1.6947   0.5279
    UE      UE270     1.2511   1.6031   0.2367


🚨 TOP 10 Iy DISCREPANCIES (Strong Axis)
Check for: Flange thickness discrepancies or depth (h) mismatches.
-------------------------------------------------------------------------------------
Family Section_ID  Area_Err%  Iy_Err%  Iz_Err%
    UE      UE180     1.3831   2.0336   0.4962
    UE  

In [7]:
# Ensure J columns exist and are numeric
active_j_audit = audit_df[
    (audit_df['DB_J'] > 0) & 
    (audit_df['Calc_J'] > 0)
].copy()

# Calculate J Error %
active_j_audit['J_Err%'] = (abs(active_j_audit['Calc_J'] - active_j_audit['DB_J']) / active_j_audit['DB_J']) * 100

cols = ['Family', 'Section_ID', 'DB_J', 'Calc_J', 'J_Err%']

print("🌪️ TOP 30 TORSIONAL CONSTANT (J) DISCREPANCIES")
print("-" * 85)
print(active_j_audit.sort_values('J_Err%', ascending=False)[cols].head(30).to_string(index=False))

print(f"\nMean J Error: {active_j_audit['J_Err%'].mean():.4f}%")

🌪️ TOP 30 TORSIONAL CONSTANT (J) DISCREPANCIES
-------------------------------------------------------------------------------------
Family Section_ID       DB_J       Calc_J    J_Err%
    UE      UE220    72700.0 4.525905e+04 37.745463
    UE      UE240    88000.0 5.690719e+04 35.332734
    UE      UE270   111000.0 7.199276e+04 35.141655
  HEAA    HE300AA   494000.0 3.225838e+05 34.699628
  HEAA    HE100AA    25100.0 1.654681e+04 34.076465
    UE      UE200    54800.0 3.625062e+04 33.849228
    UE      UE300   139000.0 9.232246e+04 33.580966
  HEAA    HE260AA   303000.0 2.014602e+05 33.511475
    HD HD260x54.1   303000.0 2.014602e+05 33.511475
  HEAA    HE320AA   559000.0 3.728451e+05 33.301405
    HD HD320x74.2   559000.0 3.728451e+05 33.301405
    UE      UE180    45600.0 3.048571e+04 33.145368
    UE      UE160    37500.0 2.531722e+04 32.487414
  HEAA    HE340AA   631000.0 4.289505e+05 32.020523
    UE      UE100    18800.0 1.281871e+04 31.815383
    UE      UE140    30400.0 2.0742

In [8]:
# Grouping by Family and describing the error percentages
family_stats = active_j_audit.groupby('Family')['J_Err%'].describe()

# Sorting by the mean error to see which family is the most "problematic"
print("📊 TORSIONAL ERROR STATISTICS BY FAMILY")
print("-" * 100)
print(family_stats.sort_values('mean', ascending=False).to_string())

# Optional: Get a simpler summary for quick reading
summary = active_j_audit.groupby('Family')['J_Err%'].agg(['mean', 'max', 'count'])
print("\n📝 QUICK SUMMARY (Mean vs Max Error)")
print("-" * 50)
print(summary.sort_values('mean', ascending=False))

📊 TORSIONAL ERROR STATISTICS BY FAMILY
----------------------------------------------------------------------------------------------------
        count       mean       std        min        25%        50%        75%        max
Family                                                                                   
UE       13.0  32.557852  2.662014  28.027062  30.748706  32.487414  33.849228  37.745463
HEAA     24.0  27.796277  4.273272  20.394161  24.518257  27.337782  31.150696  34.699628
IPN      21.0  26.827279  0.504697  25.132811  26.771257  26.989910  27.076284  27.343504
IPEA     18.0  26.499324  2.835152  21.338000  24.759859  27.703382  28.800723  30.161297
HE       41.0  26.254512  1.724387  19.895261  25.382058  26.379393  27.546688  28.799521
IPEAA     9.0  24.486270  3.991462  19.303987  20.893528  23.450952  26.914770  31.068153
HL       39.0  24.384892  2.050013  19.530381  23.268190  24.752002  25.736028  27.644278
HEA      24.0  24.078084  2.776537  19.403193  22.

In [9]:
print(sectio.catalog.get_schema())

   column_name                                    description unit
0            h                              Height of section   mm
1            b                               Width of section   mm
2           tf                               Flange thickness   mm
3           tw                                  Web thickness   mm
4       r_root                          Radius of root fillet   mm
..         ...                                            ...  ...
60       Wz,pl  Plastic modulus of section about the z-z axis  mm3
61          Ct                     Torsional modulus constant  mm3
62       r_out                         Radius of outer fillet   mm
63        r_in                         Radius of inner fillet   mm
64           D                      Outer diameter of section   mm

[65 rows x 3 columns]


In [10]:
print(sectio.catalog.get_schema('IPN'))

   column_name                                        description    unit
0            h                                  Height of section      mm
1            b                                   Width of section      mm
2           tf                                   Flange thickness      mm
3           tw                                      Web thickness      mm
4       r_root                              Radius of root fillet      mm
5        r_toe                        Radius of flange toe fillet      mm
6           ys  Distance of centre of gravity to left along y-...      mm
7            d                   Depth of straight portion of web      mm
8            G                               Mass per unit length  kg.m-1
9           AL                   Painting surface per unit length  m2.m-1
10           A                                    Area of section     mm2
11          Iy           Second moment of area about the y-y axis     mm4
12          Iz           Second moment